In [1]:
from modeling_distillemb import BertModel, BertForSequenceClassification, BertForTokenClassification
from distill_emb import DistillEmbSmall, DistillEmb
from config import DistillModelConfig, DistillEmbConfig
import torch
from transformers import AutoTokenizer, RwkvConfig, RwkvModel, AutoModel
from tokenizer import CharTokenizer
from knn_classifier import KNNTextClassifier
from data_loader import load_sentiment, load_ner_dataset, load_pos_dataset
from data_loader import load_news_dataset
import pandas as pd
from retrieval import build_json_pairs, top1_accuracy
import os
from transformers import GPT2LMHeadModel
from data_loader import *
from datasets import Dataset, DatasetDict

In [2]:
num_input_chars=12

In [3]:
tokenizer = CharTokenizer.from_pretrained(pretrained_directory="distil-emb-base")
distill_config = DistillEmbConfig.from_pretrained(pretrained_model_name_or_path="distil-emb-base")
distill_model = DistillEmb.from_pretrained(pretrained_model_name_or_path="distil-emb-base")

FileNotFoundError: [Errno 2] No such file or directory: 'distil-emb-base/tokenizer_config.json'

In [ ]:
# distill_config.distill_dropout = 0.25
config = DistillModelConfig(
    vocab_size=30522,
    hidden_size=1024,
    num_hidden_layers=3,
    num_attention_heads=8,
    intermediate_size=3072,
    max_position_embeddings=1024,
    type_vocab_size=2,
    pad_token_id=0,
    position_embedding_type="absolute",
    use_cache=True,
    classifier_dropout=None,
    hidden_dropout_prob=0.1,
    embedding_type="distill",  # 'distilemb', 'fasttext'
    encoder_type='lstm', #'lstm'
    num_input_chars=num_input_chars,  # number of characters in each token
    char_vocab_size=tokenizer.char_vocab_size,
    distill_config=distill_config,
    distill_pretrained_model_name="distil-emb-base",
    is_decoder=False
)


In [ ]:
# df, labels = load_ner_dataset()
df, labels = load_pos_dataset()
labels = list(range(max(labels) + 1))

df['text'] = df['tokens'].apply(lambda x: ' '.join(x))
# remove empty text rows
df = df[df['text'].str.strip().astype(bool)].sample(frac=1.0, random_state=42).reset_index(drop=True)

Loaded 30494 rows from masakhapos.parquet columns Index(['id', 'tokens', 'labels', 'lang', 'split'], dtype='object')


In [ ]:
labels

[np.int64(0),
 np.int64(1),
 np.int64(2),
 np.int64(3),
 np.int64(4),
 np.int64(5),
 np.int64(6),
 np.int64(7),
 np.int64(8),
 np.int64(9),
 np.int64(10),
 np.int64(11),
 np.int64(12),
 np.int64(14),
 np.int64(15),
 np.int64(16),
 np.int64(17)]

In [ ]:
df

,id,tokens,labels,lang,split,text
0,151,"[Tògán, Patrice, TALƆN, ɔ́, "", yí, afɔ, sɔ, ɖo...","[0, 10, 10, 8, 1, 16, 0, 16, 2, 16, 6, 0, 2, 0...",fon,train,"Tògán Patrice TALƆN ɔ́ "" yí afɔ sɔ ɖo tè tɔ tò..."
1,709,"[Vʋʋsma, loogr, poorẽ, ivoaryẽma, lebsa, zẽmta...","[0, 0, 2, 10, 16, 0, 0, 3, 2, 7, 10, 10, 0, 6,...",mos,train,Vʋʋsma loogr poorẽ ivoaryẽma lebsa zẽmtaar min...
2,379,"[president, Emmanuel, Macron, of, France, don,...","[0, 10, 10, 2, 10, 17, 16, 8, 16, 2, 10, 10, 1...",pcm,train,president Emmanuel Macron of France don suspen...
3,137,"[Perezida, Jovenel, wayoboraga, Haïti, kuva, m...","[0, 10, 16, 10, 16, 2, 3, 16, 2, 0, 11, 16, 1]",kin,train,Perezida Jovenel wayoboraga Haïti kuva mu 2016...
4,247,"[A, bɛ, hakilijigin, kɛ, k', a, y', a, fɔ, kab...","[11, 17, 0, 16, 7, 11, 7, 11, 16, 0, 0, 3, 1, ...",bam,test,A bɛ hakilijigin kɛ k' a y' a fɔ kabini san sa...
...,...,...,...,...,...,...
30489,99,"[Ka, wano, to, mu, ,, na, nka, hwee, .]","[16, 16, 11, 2, 1, 9, 16, 16, 1]",twi,dev,"Ka wano to mu , na nka hwee ."
30490,14,"[Erias, Lukwago, ye, mubaka, wa, Paalamenti, o...","[10, 10, 17, 0, 2, 0, 2, 10, 6, 1]",lug,train,Erias Lukwago ye mubaka wa Paalamenti owa Kamp...
30491,85,"[Daʼgaə́, mthə́dzə, ntʉ́m, gɔ̂pnaʼ, pə́, wə́, ...","[9, 0, 2, 0, 17, 17, 16, 0, 7, 16, 10, 7, 16, ...",bbj,train,Daʼgaə́ mthə́dzə ntʉ́m gɔ̂pnaʼ pə́ wə́ jɔ́ mjy...
30492,532,"[ƐFIYƐSIDE, ,, hali, n', a, waatilaɲɛmɔgɔya, t...","[0, 1, 0, 5, 11, 0, 7, 17, 10, 10, 10, 0, 1, 0...",bam,test,"ƐFIYƐSIDE , hali n' a waatilaɲɛmɔgɔya tun bɛ S..."


In [ ]:
lang_counts = df.groupby('split')['lang'].nunique()
for split, count in lang_counts.items():
    print(f"{split.capitalize()} split has {count} languages.")

Dev split has 20 languages.
Test split has 20 languages.
Train split has 20 languages.


In [ ]:
label2id = {label.item(): idx for idx, label in enumerate(labels)}
id2label = {idx: label for label, idx in label2id.items()}
config.label2id = label2id
config.id2label = id2label

print(f"Converted labels to integers: {label2id}")
print(f"Converted integers to labels: {id2label}")

Converted labels to integers: {0: 0, 1: 1, 2: 2, 3: 3, 4: 4, 5: 5, 6: 6, 7: 7, 8: 8, 9: 9, 10: 10, 11: 11, 12: 12, 14: 13, 15: 14, 16: 15, 17: 16}
Converted integers to labels: {0: 0, 1: 1, 2: 2, 3: 3, 4: 4, 5: 5, 6: 6, 7: 7, 8: 8, 9: 9, 10: 10, 11: 11, 12: 12, 13: 14, 14: 15, 15: 16, 16: 17}


In [ ]:
config.num_labels = len(labels)
model = BertForTokenClassification(config)

In [ ]:

train_df = df[df['split'] == 'train']
test_df = df[df['split'] == 'test']

In [ ]:

# Create HuggingFace datasets
train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)
train_dataset

Dataset({
    features: ['id', 'tokens', 'labels', 'lang', 'split', 'text', '__index_level_0__'],
    num_rows: 15263
})

In [ ]:
from typing import Dict, Any

def preprocess_function(examples: Dict[str, Any]):
    batch = tokenizer(
        examples["text"],
        padding=False,
        max_length=512,
        return_attention_mask=False,
    )

    batch["labels"] = examples["labels"]
    return batch

tokenized_train = train_dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=train_dataset.column_names,
)

tokenized_test = test_dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=test_dataset.column_names,
)

Map:   0%|          | 0/15263 [00:00<?, ? examples/s]

Map:   0%|          | 0/12190 [00:00<?, ? examples/s]

In [ ]:
from transformers import Trainer, TrainingArguments, DataCollatorWithPadding
class CustomDataCollator:
    def __init__(self, tokenizer):
        self.tokenizer = tokenizer

    def __call__(self, features):
        batch = self.tokenizer.pad(
            features,
            padding="longest",
            max_length=512,
            return_tensors="pt",
            return_attention_mask=True,
        )
        
        max_len = batch["input_ids"].shape[1] - 2  # exclude special tokens
        padded_labels = []
        for f in features:
            label = f["labels"]
            padded_label = [-100] +  label + [-100] * (max_len - len(label)) + [-100]
            padded_labels.append(padded_label)
        batch["labels"] = torch.tensor(padded_labels, dtype=torch.long)
        assert batch["labels"].shape == (batch["input_ids"].shape[0], batch["input_ids"].shape[1]), f"Labels shape {batch['labels'].shape} does not match input_ids shape {batch['input_ids'].shape}"
        return batch

data_collator = CustomDataCollator(tokenizer)

In [ ]:
##### from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
import numpy as np
from sklearn.metrics import accuracy_score, f1_score
from seqeval.metrics import accuracy_score as seqeval_accuracy_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    # Remove ignored index (special tokens) and convert to labels
    true_labels_seq = [[id2label[l.item()] for l in label if l != -100] for label in labels]
    true_predictions_seq = [
        [id2label[p.item()] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    
    # Compute seqeval metrics
    seqeval_results = seqeval_accuracy_score(true_labels_seq, true_predictions_seq)

    # Flatten labels and predictions for token-level metrics
    true_labels_flat = []
    pred_labels_flat = []
    for pred_seq, label_seq in zip(predictions, labels):
        mask = label_seq != -100
        true_labels_flat.extend(label_seq[mask])
        pred_labels_flat.extend(pred_seq[mask])

    label_ids = list(label2id.values())

    # Compute token-level metrics
    token_metrics = {
        "accuracy": accuracy_score(true_labels_flat, pred_labels_flat),
        "f1_weighted": f1_score(true_labels_flat, pred_labels_flat, average="weighted", labels=label_ids, zero_division=0),
        "f1_macro": f1_score(true_labels_flat, pred_labels_flat, average="macro", labels=label_ids, zero_division=0),
        "f1_micro": f1_score(true_labels_flat, pred_labels_flat, average="micro", labels=label_ids, zero_division=0),
        'seqeval_accuracy': seqeval_results
    }
    
    return token_metrics


import os
from datasets import load_metric
dataloader_num_workers=os.cpu_count() - 1
batch_size = 32

training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=3e-4,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    num_train_epochs=20,
    weight_decay=0.0,
    report_to=[],
    eval_strategy="epoch",  
    save_total_limit=1,
    save_only_model=True,
    logging_strategy="steps",
    logging_steps=10,
    label_smoothing_factor=0.1,
    max_grad_norm=1.0,
    warmup_ratio=0.0,
    lr_scheduler_type="cosine",
    dataloader_num_workers=4,        # Number of CPU workers for data loading
    dataloader_pin_memory=True,      # Faster GPU transfer
    gradient_accumulation_steps=1
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

# Evaluate the model after training
eval_results = trainer.evaluate()
print(f"Evaluation results: {eval_results}")

/pytorch/aten/src/ATen/native/cuda/ScatterGatherKernel.cu:163: operator(): block: [2,0,0], thread: [42,0,0] Assertion `idx_dim >= 0 && idx_dim < index_size && "scatter gather kernel index out of bounds"` failed.
/pytorch/aten/src/ATen/native/cuda/ScatterGatherKernel.cu:163: operator(): block: [2,0,0], thread: [11,0,0] Assertion `idx_dim >= 0 && idx_dim < index_size && "scatter gather kernel index out of bounds"` failed.
/pytorch/aten/src/ATen/native/cuda/ScatterGatherKernel.cu:163: operator(): block: [1,0,0], thread: [2,0,0] Assertion `idx_dim >= 0 && idx_dim < index_size && "scatter gather kernel index out of bounds"` failed.
/pytorch/aten/src/ATen/native/cuda/ScatterGatherKernel.cu:163: operator(): block: [0,0,0], thread: [6,0,0] Assertion `idx_dim >= 0 && idx_dim < index_size && "scatter gather kernel index out of bounds"` failed.
/pytorch/aten/src/ATen/native/cuda/ScatterGatherKernel.cu:163: operator(): block: [1,0,0], thread: [40,0,0] Assertion `idx_dim >= 0 && idx_dim < index_siz

AcceleratorError: CUDA error: device-side assert triggered
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [ ]:
from collections import defaultdict

pred_output = trainer.predict(tokenized_test)
logits = pred_output.predictions
label_ids = pred_output.label_ids
langs = test_df["lang"].tolist()

lang_true, lang_pred = defaultdict(list), defaultdict(list)
for idx, lang in enumerate(langs):
    label_seq = label_ids[idx]
    pred_seq = logits[idx].argmax(axis=-1)
    mask = label_seq != -100
    if not np.any(mask):
        continue
    lang_true[lang].extend(label_seq[mask])
    lang_pred[lang].extend(pred_seq[mask])

label_id_list = list(label2id.values())
lang_metrics = {}
for lang, true_values in lang_true.items():
    preds = lang_pred[lang]
    lang_metrics[lang] = {
        "accuracy": accuracy_score(true_values, preds),
        "f1_weighted": f1_score(true_values, preds, average="weighted", labels=label_id_list, zero_division=0),
        "f1_macro": f1_score(true_values, preds, average="macro", labels=label_id_list, zero_division=0),
        "f1_micro": f1_score(true_values, preds, average="micro", labels=label_id_list, zero_division=0),
        "num_tokens": len(true_values),
    }

lang_metrics_df = pd.DataFrame.from_dict(lang_metrics, orient="index").sort_values("f1_macro", ascending=False)
lang_metrics_df

In [ ]:
lang_metrics_df = pd.DataFrame.from_dict(lang_metrics, orient="index").sort_index()
lang_metrics_df

In [ ]:
lang_metrics_df['f1_macro'].mean()

In [4]:
# 89.2 77.8 87.5 82.4 92.7 77.8 97.4 90.8 86.8 89.6 81.1 89.5 88.7 92.8 83.8 83.9 92.1 87.5 91.1 88.8
mean = sum([89.2, 77.8, 87.5, 82.4, 92.7, 77.8, 97.4, 90.8, 86.8, 89.6, 81.1, 89.5, 88.7, 92.8, 83.8, 83.9, 92.1, 87.5, 91.1, 88.8]) / 20

In [5]:
mean

87.56499999999998